In [ ]:
%pip install -q -U transformers accelerate bitsandbytes


In [ ]:
import json
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

INPUT_JSON = Path("/content/graph.json")
OUTPUT_JSON = INPUT_JSON.with_name(INPUT_JSON.stem + "_answers.json")
MODEL_ID = "allenai/Olmo-3-7B-Instruct"
MAX_CONTEXT_TOKENS = 10_000
MAX_NEW_TOKENS = 2048

payload = json.loads(INPUT_JSON.read_text(encoding="utf-8"))
SYSTEM_PROMPT = payload["system_prompt"]
USER_MESSAGE_TEMPLATE = payload.get("user_message_template", "{context}{question}")


In [ ]:
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map={"": 0},
    quantization_config=quantization_config,
    dtype=compute_dtype,
    attn_implementation="sdpa",
)
model.eval()


In [ ]:
results = {
    "model_id": MODEL_ID,
    "case_id": payload["case_id"],
    "variant": payload["variant"],
    "system_prompt": SYSTEM_PROMPT,
    "user_message_template": USER_MESSAGE_TEMPLATE,
    "quantization": {"bits": 4, "type": "nf4", "double_quant": True, "compute_dtype": str(compute_dtype)},
    "generation": {
        "max_context_tokens": MAX_CONTEXT_TOKENS,
        "do_sample": False,
        "max_new_tokens": MAX_NEW_TOKENS,
    },
    "answers": [],
}

for question in payload["questions"]:
    retrieved = sorted(question["chunks"], key=lambda chunk: chunk["rank"])
    chosen = []
    fragments = []
    truncated_chunk_ids = []
    prefix = "Context extracted from a blind search over the corpus:\n\n"
    suffix = "\n\n---\n\n"
    for chunk in retrieved:
        label = f"[Fragment {len(chosen) + 1}] [Source: {chunk['chunk_id']}]"
        if chunk.get("section"):
            label += f" [Section: {chunk['section']}]"
        def context_with(text):
            return prefix + "\n\n".join([*fragments, f"{label}\n{text}"]) + suffix
        text = chunk["text"]
        if len(tokenizer.encode(context_with(text))) > MAX_CONTEXT_TOKENS:
            if not chosen:
                low, high = 0, len(text)
                while low < high:
                    mid = (low + high + 1) // 2
                    if len(tokenizer.encode(context_with(text[:mid]))) <= MAX_CONTEXT_TOKENS:
                        low = mid
                    else:
                        high = mid - 1
                if low and len(tokenizer.encode(context_with(text[:low]))) <= MAX_CONTEXT_TOKENS:
                    fragments.append(f"{label}\n{text[:low]}")
                    chosen.append(chunk)
                    truncated_chunk_ids.append(chunk["chunk_id"])
            break
        fragments.append(f"{label}\n{text}")
        chosen.append(chunk)
    context = prefix + "\n\n".join(fragments) + suffix if fragments else ""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_MESSAGE_TEMPLATE.format(
            question=question["question"],
            context=context or ("No reference excerpts were retrieved." if "user_message_template" in payload else ""),
        )},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            do_sample=False,
            max_new_tokens=MAX_NEW_TOKENS,
            pad_token_id=tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id,
        )
    input_length = inputs.input_ids.shape[1]
    answer = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True).strip()
    results["answers"].append({
        "id": question["id"],
        "question": question["question"],
        "answer": answer,
        "expected_answer": question.get("expected_answer"),
        "chunk_ids": [chunk["chunk_id"] for chunk in chosen],
        "omitted_chunk_ids": [chunk["chunk_id"] for chunk in retrieved[len(chosen):]],
        "truncated_chunk_ids": truncated_chunk_ids,
        "context_tokens": len(tokenizer.encode(context)) if context else 0,
        "input_tokens": input_length,
        "output_tokens": outputs.shape[1] - input_length,
    })
    OUTPUT_JSON.write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")
    print(question["id"], answer, sep="\n", end="\n\n")
    del inputs, outputs
